# Batch 5D — Revision 5 Frontier API Baseline

This notebook runs the pre-committed 100-row Revision 5 comparison between frozen ComplianceGPT outputs and the stable Gemini API model `gemini-3.5-flash`. It preserves the exact questions, ordered evidence windows, free-form prompt/parser, ASK policy, and offline verifier used in the registered design.

A GPU is not needed. Use a CPU runtime because inference is remote.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Before running

In Colab's **Secrets** panel (key icon), add the Paid Tier 1 API key as `GEMINI_API_KEY` and enable notebook access. The notebook never prints or saves the key. Confirm that the key belongs to the billed Google project before starting the API cell.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/Wenjia1215/ComplianceGPT.git'
SOURCE_COMMIT = 'b7e10b0993147378ade09853265ac8af1214a729'
REPO_DIR = Path('/content/ComplianceGPT_batch5d')
OUTPUT_DIR = Path('/content/drive/MyDrive/rq2_frontier_baseline_rev5_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / '.git').is_dir():
    dirty = subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError(
            f'{REPO_DIR} contains local changes. Start a fresh runtime or move that directory.'
        )
else:
    subprocess.run(
        ['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(REPO_DIR)],
        check=True,
    )
subprocess.run(['git', 'fetch', 'origin', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
checked_out = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
assert checked_out == SOURCE_COMMIT, (checked_out, SOURCE_COMMIT)
print('Pinned Batch 5D source:', checked_out)
print('Persistent result directory:', OUTPUT_DIR)

In [ ]:
import json
from importlib.metadata import version

runtime_manifest = OUTPUT_DIR / 'manifests/api_runtime.json'
packages = ['numpy==2.0.2', 'PyYAML==6.0.3']
if runtime_manifest.is_file():
    prior_runtime = json.loads(runtime_manifest.read_text(encoding='utf-8'))
    prior_google_genai = str(
        prior_runtime.get('google_genai_python', '') or ''
    ).strip()
    if prior_google_genai:
        packages.append(f'google-genai=={prior_google_genai}')
        print('Resuming with recorded Google Gen AI SDK:', prior_google_genai)
    else:
        packages.append('google-genai>=2.25.0,<3')
else:
    packages.append('google-genai>=2.25.0,<3')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *packages], check=True
)
print('Google Gen AI SDK:', version('google-genai'))

In [ ]:
import os
from google.colab import userdata

try:
    api_key = str(userdata.get('GEMINI_API_KEY') or '').strip()
except Exception as exc:
    raise RuntimeError(
        'Add GEMINI_API_KEY in Colab Secrets (key icon), enable notebook access, then rerun this cell.'
    ) from exc
if not api_key:
    raise RuntimeError('GEMINI_API_KEY is empty in Colab Secrets.')
os.environ['GEMINI_API_KEY'] = api_key
del api_key
print('GEMINI_API_KEY loaded into process memory; its value was not displayed or written.')

In [ ]:
test_environment = os.environ.copy()
test_environment['PYTHONPATH'] = str(REPO_DIR / 'src')
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_directory = REPO_DIR / 'tests'
required_tests = (
    'test_frontier_api_answerer.py',
    'test_rq2_frontier_baseline.py',
    'test_rq2_frontier_baseline_rev5.py',
)
for test_name in required_tests:
    if not (test_directory / test_name).is_file():
        raise FileNotFoundError(f'Batch 5D test file is missing: {test_directory / test_name}')
test_command = [
    sys.executable, '-m', 'unittest', 'discover',
    '-s', str(test_directory), '-p', 'test_*frontier*.py', '-v',
]
test_result = subprocess.run(
    test_command, cwd=REPO_DIR, env=test_environment,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(test_result.stdout, end='')
if test_result.returncode != 0:
    raise RuntimeError(
        f'Batch 5D offline tests failed (exit {test_result.returncode}). '
        'See the unittest output above for the underlying error.'
    )
print('Batch 5D offline tests passed.')

In [ ]:
import hashlib
from IPython.display import Markdown, display

FROZEN_ARCHIVE_SHA256 = '56a70db6eca0420df6affbe63c859418ac423b54d80d3d1eae7bf785f0f63328'
PRECOMMITMENT_SHA256 = 'b0cc78debb25c1025f05622348925383b7a19503751680e1a6a0b3ddefa34504'
frozen_archive = (
    REPO_DIR / 'experiments/answerer_comparison/rq2_matched/results_v3/compliancegpt_rq2_matched_v3.zip'
)
precommitment = (
    REPO_DIR / 'experiments/answerer_comparison/rq2_frontier_baseline_rev5/precommitment.json'
)
actual_archive_sha256 = hashlib.sha256(frozen_archive.read_bytes()).hexdigest()
actual_precommitment_sha256 = hashlib.sha256(precommitment.read_bytes()).hexdigest()
assert actual_archive_sha256 == FROZEN_ARCHIVE_SHA256
assert actual_precommitment_sha256 == PRECOMMITMENT_SHA256
runner = (
    REPO_DIR / 'experiments/answerer_comparison/rq2_frontier_baseline_rev5/run_frontier_baseline_rev5.py'
)
prepare_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--prepare-only',
]
subprocess.run(prepare_command, cwd=REPO_DIR, check=True)
contexts_path = OUTPUT_DIR / 'contexts/rev5_prepared_contexts.jsonl'
with contexts_path.open('r', encoding='utf-8') as handle:
    prepared_rows = sum(1 for line in handle if line.strip())
assert prepared_rows == 100, f'Expected 100 prepared contexts, found {prepared_rows}'
preflight_summary = json.loads(
    (OUTPUT_DIR / 'preflight/summary.json').read_text(encoding='utf-8')
)
assert preflight_summary.get('scope') == 'gold-free frozen-context integrity audit; no API call'
assert preflight_summary.get('api_calls_performed') == 0
print('Frozen archive verified:', actual_archive_sha256)
print('Pre-commitment verified:', actual_precommitment_sha256)
print(f'Prepared-context audit passed: {prepared_rows} Revision 5 contexts; no API call made.')
display(Markdown((OUTPUT_DIR / 'PREPARED_CONTEXTS.md').read_text(encoding='utf-8')))

## Run the registered API comparison

This cell must complete all 100 rows. Every returned API response and validated row is checkpointed. A transient per-minute 429 waits and resumes automatically; authentication errors, daily-quota failures, and other errors stop for inspection. After a runtime disconnect, reconnect and use **Runtime → Run all**.

In [ ]:
from collections import deque
import math
import re
import time

def per_minute_quota_wait_seconds(output_text):
    if '429 RESOURCE_EXHAUSTED' not in output_text:
        return None
    if 'PerMinute' not in output_text and 'requests per minute' not in output_text.lower():
        return None
    matches = []
    for pattern in (r'Please retry in\s+([0-9]+(?:\.[0-9]+)?)s',):
        matches.extend(float(value) for value in re.findall(pattern, output_text))
    server_delay = max(matches, default=60.0)
    return max(65, math.ceil(server_delay + 5.0))

run_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
]
run_environment = os.environ.copy()
maximum_quota_restarts = 50
quota_restarts = 0
failure_log = OUTPUT_DIR / 'runner_failure_tail.log'
print('Starting the 100-row Revision 5 Gemini 3.5 Flash API baseline.')
print('Transient per-minute 429 responses will wait and resume automatically.')
print('After a disconnect, reconnect and Run all; validated rows are skipped.')
while True:
    tail = deque(maxlen=400)
    process = subprocess.Popen(
        run_command, cwd=REPO_DIR, env=run_environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        tail.append(line)
    return_code = process.wait()
    if return_code == 0:
        break
    failure_output = ''.join(tail)
    failure_log.write_text(failure_output, encoding='utf-8')
    wait_seconds = per_minute_quota_wait_seconds(failure_output)
    if wait_seconds is None or quota_restarts >= maximum_quota_restarts:
        print(f'Batch 5D failed. The final runner output is saved at {failure_log}.')
        raise subprocess.CalledProcessError(return_code, run_command)
    quota_restarts += 1
    print(
        f'Per-minute Gemini quota reached; waiting {wait_seconds}s before resumable restart '
        f'{quota_restarts}/{maximum_quota_restarts}. Completed rows remain checkpointed.'
    )
    time.sleep(wait_seconds)
if failure_log.exists():
    failure_log.unlink()
print('Batch 5D runner completed successfully.')

In [ ]:
summary_path = OUTPUT_DIR / 'summary.json'
api_manifest_path = OUTPUT_DIR / 'manifests/api_responses.json'
assert summary_path.is_file(), summary_path
assert api_manifest_path.is_file(), api_manifest_path
summary = json.loads(summary_path.read_text(encoding='utf-8'))
api_manifest = json.loads(api_manifest_path.read_text(encoding='utf-8'))
assert summary.get('result_id') == 'rq2_frontier_baseline_rev5_v1'
assert summary.get('n_questions') == 100
assert api_manifest.get('completed_result_rows') == 100
display(Markdown((OUTPUT_DIR / 'SUMMARY.md').read_text(encoding='utf-8')))
print('Server-reported model:', api_manifest['response_model'])
print('All logged API usage:', api_manifest['all_logged_usage']['totals'])
print('Parse-retry rows:', api_manifest['parse_retry_rows'])
print('Orphaned API calls:', api_manifest['orphaned_call_count'])

In [ ]:
archive_path = OUTPUT_DIR.parent / f'{OUTPUT_DIR.name}.zip'
assert archive_path.is_file(), f'Result ZIP was not created: {archive_path}'
archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
print('Confirmed result directory:', OUTPUT_DIR)
print('Confirmed result ZIP:', archive_path)
print('Result ZIP SHA-256:', archive_sha256)
print('Download this executed notebook and provide it with the ZIP for audit.')

## Handoff

After the final cell succeeds, download this executed `.ipynb` from Colab and provide it together with `/content/drive/MyDrive/rq2_frontier_baseline_rev5_v1.zip`. Do not edit result rows or summary values. The executed notebook, raw API log, contracts, manifests, summary, and hashes are required for the post-run audit and repository archive.